In [48]:
import json
import os
import gzip
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime

from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

## DuckDB reading of data (credit to Ilya M)

In [49]:
DATA_DIR = Path("../data/raw")
CATEGORY = "Musical_Instruments"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"

OUTPUT_DIR = Path("../data/processed")
OUTPUT_FILE  = OUTPUT_DIR / f"{CATEGORY}_merged.parquet"


In [50]:
c2 = duckdb.connect()

In [51]:
# Reviews File
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Five Stars,"Great headphones, comfortable and sound is goo...",[],B003LPTAYI,B003LPTAYI,AGKASBHYZPGTEPO6LWZPVJWB2BVA,1452650586000,0,True
1,3.0,nice sound. pedal failed after less than 1 year,I like the piano.. but the sustain pedal faile...,[],B00723436A,B06XP6TDVY,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1558567365290,2,True
2,4.0,okay,pretty good overall. I like it. the controll...,[],B0040FJ27S,B0040FJ27S,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1384912482000,0,True
3,3.0,Easy to return.,Too bad it didn't work. At least the return pr...,[],B00191WVF6,B00WJ3HL5I,AEM663T6XHZFWLODF4US2RCOCUSA,1607055693671,0,True
4,5.0,Good product despite tight bolt.,Good and sturdy but the bolt was hell to get o...,[],B07T9NM5QR,B07T9NM5QR,AFJTRBXMURLHS5EGNXLUHDHIZRFQ,1622595785255,0,False


In [52]:
# Metadata Files
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Musical Instruments,Pearl Export Lacquer EXL725S/C249 5-Piece New ...,4.2,22,[Item may ship in more than one box and may ar...,[Introducing the best selling drum set of all ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Best Selling Drum Set of All Time'...,Pearl,"[Musical Instruments, Drums & Percussion, Drum...","{'Item Weight': '""33 pounds""', 'Product Dimens...",B01M4HO6RK,None,None,<NA>
1,Musical Instruments,Behringer EUROPOWER EPQ900 Professional 900 Wa...,4.0,13,[2 x 390 Watts into 4 Ohms; 2 x 245 Watts into...,"[BEHRINGER EUROPOWER EPQ900, Professional 900-...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Behringer,"[Musical Instruments, Live Sound & Stage, Powe...","{'Item Weight': '""10.8 pounds""', 'Product Dime...",B00508JFE4,None,None,<NA>
2,Musical Instruments,Washburn Classical Series Acoustic Electric Cu...,3.6,15,[The Washburn is truly a professional instrume...,[C64SCE CLASSICAL GUITAR The cutaway allows ac...,399.0,[{'thumb': 'https://m.media-amazon.com/images/...,[],Washburn,"[Musical Instruments, Guitars]","{'Item Weight': '""5.98 pounds""', 'Product Dime...",B000S5JGMU,None,None,<NA>
3,Musical Instruments,"VocoPro, plug in, Black, 21.00 x 21.00 x 23.00...",3.5,7,"[Includes one microphone and one receiver, Can...",[VocoPro UHF-18 DIAMOND - N Wireless Microphon...,112.0,[{'thumb': 'https://m.media-amazon.com/images/...,[],VocoPro,"[Musical Instruments, Live Sound & Stage, PA S...","{'Item Weight': '""2.29 pounds""', 'Product Dime...",B00B2HLWZW,None,None,<NA>
4,Musical Instruments,Shure SM7B Vocal Dynamic Microphone for Broadc...,4.9,9512,[ONE MICROPHONE FOR EVERYTHING - Studio Record...,"[The SM7B dynamic microphone has a smooth, fla...",399.0,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Shure SM7B Mic Demonstration', 'ur...",Shure,"[Musical Instruments, Microphones & Accessorie...","{'Item Weight': '""2.7 pounds""', 'Product Dimen...",B0B89ZSYS7,None,None,<NA>


In [53]:
# Reviews schema
review_schema = c2.execute(f"""
    DESCRIBE SELECT * FROM read_json_auto('{REVIEWS_URL}')
""").df()
print("=== REVIEWS SCHEMA ===")
print(review_schema.to_string(index=False))

=== REVIEWS SCHEMA ===
      column_name                                                                                                   column_type null  key default extra
           rating                                                                                                        DOUBLE  YES None    None  None
            title                                                                                                       VARCHAR  YES None    None  None
             text                                                                                                       VARCHAR  YES None    None  None
           images STRUCT(small_image_url VARCHAR, medium_image_url VARCHAR, large_image_url VARCHAR, attachment_type VARCHAR)[]  YES None    None  None
             asin                                                                                                       VARCHAR  YES None    None  None
      parent_asin                                                

In [54]:
# Reviews schema
meta_schema = c2.execute(f"""
    DESCRIBE SELECT * FROM read_json_auto('{META_URL}')
""").df()
print("=== MetaData Schema ===")
print(meta_schema.to_string(index=False))

=== MetaData Schema ===
    column_name                                                               column_type null  key default extra
  main_category                                                                   VARCHAR  YES None    None  None
          title                                                                   VARCHAR  YES None    None  None
 average_rating                                                                    DOUBLE  YES None    None  None
  rating_number                                                                    BIGINT  YES None    None  None
       features                                                                 VARCHAR[]  YES None    None  None
    description                                                                 VARCHAR[]  YES None    None  None
          price                                                                      JSON  YES None    None  None
         images STRUCT(thumb VARCHAR, "large" VARCHAR, variant V

In [55]:
size_stats = c2.execute(f"""
    SELECT
        (SELECT COUNT(*) FROM read_json_auto('{REVIEWS_URL}')) AS total_reviews,
        (SELECT COUNT(*) FROM read_json_auto('{META_URL}'))    AS total_products,
        (SELECT COUNT(DISTINCT parent_asin) FROM read_json_auto('{REVIEWS_URL}')) AS unique_products_reviewed
""").df()
print(size_stats)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_reviews  total_products  unique_products_reviewed
0        3017439          213593                    213571


In [56]:
# Download and save reviews dataset
if not REVIEWS_FILE.exists():
    print(f"Downloading reviews from {REVIEWS_URL}...")
    urllib.request.urlretrieve(REVIEWS_URL, REVIEWS_FILE)
else:
    print("Reviews file already exists.")

print("Loading reviews data...")
reviews_df = pd.read_json(REVIEWS_FILE, lines=True, compression='gzip')

reviews_parquet = OUTPUT_DIR / "reviews.parquet"
print(f"Saving reviews to {reviews_parquet}...")
reviews_df.to_parquet(reviews_parquet)

# Download and save meta dataset
if not META_FILE.exists():
    print(f"Downloading meta from {META_URL}...")
    urllib.request.urlretrieve(META_URL, META_FILE)
else:
    print("Meta file already exists.")

print("Loading meta data...")
meta_df = pd.read_json(META_FILE, lines=True, compression='gzip')

# Handle mixed types in price column
meta_df['price'] = meta_df['price'].astype(str)

meta_parquet = OUTPUT_DIR / "meta.parquet"
print(f"Saving meta to {meta_parquet}...")
meta_df.to_parquet(meta_parquet)

print("Done!")

Reviews file already exists.
Loading reviews data...
Saving reviews to ..\data\processed\reviews.parquet...
Meta file already exists.
Loading meta data...
Saving meta to ..\data\processed\meta.parquet...
Done!


## Selection and Justification of fields for retrieval

Based on the possible fields, we decided to keep the following fields in the dataframe because they have the most semantically rich information that would be useful for querying.  From EDA, there is no NA values as well which implies data quality for analysis. 

| File | Field | Type | Description |
|---|---|---|---|
| Review | `rating` | float | 1.0–5.0 star rating |
| Review | `title` | str | Review headline |
| Review | `text` | str | Full review body |
| Review | `parent_asin` | str | Parent product ID (use to join metadata) |
| Review | `verified_purchase` | bool | Verified buyer flag |
| Review | `helpful_vote` | int | Upvotes on the review |
| Meta | `title` | str | Product name |
| Meta | `average_rating` | float | Aggregate product rating |
| Meta | `features` | list[str] | Bullet-point features |
| Meta | `description` | list[str] | Product description paragraphs |
| Meta | `parent_asin` | str | Join key to reviews |

## Preprocess & Save to Processed Subfolder

In [57]:
meta = pd.read_parquet(OUTPUT_DIR / "meta.parquet", columns=['parent_asin', 'title', 'average_rating', 'features', 'description'], engine = 'pyarrow')
meta.head()

,parent_asin,title,average_rating,features,description
0,B01M4HO6RK,Pearl Export Lacquer EXL725S/C249 5-Piece New ...,4.2,[Item may ship in more than one box and may ar...,[Introducing the best selling drum set of all ...
1,B00508JFE4,Behringer EUROPOWER EPQ900 Professional 900 Wa...,4.0,[2 x 390 Watts into 4 Ohms; 2 x 245 Watts into...,"[BEHRINGER EUROPOWER EPQ900, Professional 900-..."
2,B000S5JGMU,Washburn Classical Series Acoustic Electric Cu...,3.6,[The Washburn is truly a professional instrume...,[C64SCE CLASSICAL GUITAR The cutaway allows ac...
3,B00B2HLWZW,"VocoPro, plug in, Black, 21.00 x 21.00 x 23.00...",3.5,"[Includes one microphone and one receiver, Can...",[VocoPro UHF-18 DIAMOND - N Wireless Microphon...
4,B0B89ZSYS7,Shure SM7B Vocal Dynamic Microphone for Broadc...,4.9,[ONE MICROPHONE FOR EVERYTHING - Studio Record...,"[The SM7B dynamic microphone has a smooth, fla..."


In [58]:
reviews = pd.read_parquet(OUTPUT_DIR / "reviews.parquet", columns =['rating', 'title', 'text', 'parent_asin', 'helpful_vote', 'verified_purchase'], engine = 'pyarrow')
reviews.head()

,rating,title,text,parent_asin,helpful_vote,verified_purchase
0,5,Five Stars,"Great headphones, comfortable and sound is goo...",B003LPTAYI,0,True
1,3,nice sound. pedal failed after less than 1 year,I like the piano.. but the sustain pedal faile...,B06XP6TDVY,2,True
2,4,okay,pretty good overall. I like it. the controll...,B0040FJ27S,0,True
3,3,Easy to return.,Too bad it didn't work. At least the return pr...,B00WJ3HL5I,0,True
4,5,Good product despite tight bolt.,Good and sturdy but the bolt was hell to get o...,B07T9NM5QR,0,False


In [59]:
print(f"{meta.info()}")
print(f"{reviews.info()}")

<class 'pandas.DataFrame'>
RangeIndex: 213593 entries, 0 to 213592
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   parent_asin     213593 non-null  str    
 1   title           213593 non-null  str    
 2   average_rating  213593 non-null  float64
 3   features        213593 non-null  object 
 4   description     213593 non-null  object 
dtypes: float64(1), object(2), str(2)
memory usage: 28.3+ MB
None
<class 'pandas.DataFrame'>
RangeIndex: 3017439 entries, 0 to 3017438
Data columns (total 6 columns):
 #   Column             Dtype
---  ------             -----
 0   rating             int64
 1   title              str  
 2   text               str  
 3   parent_asin        str  
 4   helpful_vote       int64
 5   verified_purchase  bool 
dtypes: bool(1), int64(2), str(3)
memory usage: 902.6 MB
None


In [60]:
meta.isna().sum()

parent_asin       0
title             0
average_rating    0
features          0
description       0
dtype: int64

In [61]:
reviews.isna().sum()

rating               0
title                0
text                 0
parent_asin          0
helpful_vote         0
verified_purchase    0
dtype: int64

## Export Sample & Preprocessing Function

In [71]:
def filter_reviews(df: pd.DataFrame) -> pd.DataFrame:
    # Keep verified purchases only
    df = df[df['verified_purchase'] == True].copy()

    # Keep ratings >= 3.0
    df = df[df['rating'] >= 3.0]

    # Combine title and text
    df['text'] = df['title'].fillna('') + ': ' + df['text'].fillna('')

    # Drop the title column
    df = df.drop(columns=['title'])

    # Keep texts between 50 and 300 characters
    df = df[df['text'].str.len().between(50, 300)]

    return df

In [72]:
reviews_filtered = filter_reviews(reviews)
print(f"{reviews_filtered.info()}")
reviews_filtered.head()


<class 'pandas.DataFrame'>
Index: 1425340 entries, 0 to 3017438
Data columns (total 5 columns):
 #   Column             Non-Null Count    Dtype
---  ------             --------------    -----
 0   rating             1425340 non-null  int64
 1   text               1425340 non-null  str  
 2   parent_asin        1425340 non-null  str  
 3   helpful_vote       1425340 non-null  int64
 4   verified_purchase  1425340 non-null  bool 
dtypes: bool(1), int64(2), str(2)
memory usage: 263.7 MB
None


,rating,text,parent_asin,helpful_vote,verified_purchase
0,5,"Five Stars: Great headphones, comfortable and ...",B003LPTAYI,0,True
1,3,nice sound. pedal failed after less than 1 ye...,B06XP6TDVY,2,True
2,4,okay: pretty good overall. I like it. the co...,B0040FJ27S,0,True
3,3,Easy to return.: Too bad it didn't work. At le...,B00WJ3HL5I,0,True
6,5,Works great for thinner books.: Xlnt quality. ...,B07Z9M4RS1,1,True


In [76]:
sample_fraction = 0.72

sampled_reviews = reviews_filtered.sample(frac=sample_fraction, random_state=42)
len(sampled_reviews)

1026245

In [77]:
filtered_reviews_parquet = OUTPUT_DIR / "filtered_reviews.parquet"

if filtered_reviews_parquet.exists():
    print(f"File already exists: {filtered_reviews_parquet}")
else:
    sampled_reviews.to_parquet(filtered_reviews_parquet, index=False)
    size_mb = os.path.getsize(filtered_reviews_parquet) / (1024 ** 2)
    print(f"✅ Saved {len(sampled_reviews):,} rows ({size_mb:.1f} MB) to {filtered_reviews_parquet}")

✅ Saved 1,026,245 rows (94.4 MB) to ..\data\processed\filtered_reviews.parquet


In [78]:
# Filter meta to only include ASINs that exist in the sampled reviews
shared_asins = set(sampled_reviews['parent_asin'])
meta_filtered = meta[meta['parent_asin'].isin(shared_asins)]

print(f"meta rows % after alignment: {len(meta_filtered)/ len(meta):.2%}")

# Save
meta_parquet = OUTPUT_DIR / "filtered_meta.parquet"

if meta_parquet.exists():
    print(f"File already exists: {meta_parquet}")
else:
    meta_filtered.to_parquet(meta_parquet, index=False)
    size_mb = os.path.getsize(meta_parquet) / (1024 ** 2)
    print(f"✅ Saved {len(meta_filtered):,} rows ({size_mb:.1f} MB) to {meta_parquet}")

meta rows % after alignment: 59.83%
✅ Saved 127,800 rows (76.9 MB) to ..\data\processed\filtered_meta.parquet


## Description of text preprocessing decisions:

As shown in above, we are cleaning the corpus to:
- keep only reviews that are greater than 3 / 5
- keep reviews only with a verified purchase
- Only keep reviews where text is greater than 50 words but less than 300. The thinking is that for very short or long reviews, the quality of the reviews will decrease. 
- combined the text and title column to avoid redundancy

This will allow our models to optimally understand and identify our corpus. 